<a href="https://colab.research.google.com/github/Rithan377/GAZE-Estimation/blob/main/GAZELLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- SETUP ---
!pip install retina-face opencv-python-headless -q

import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import requests
from io import BytesIO
import numpy as np
import cv2
from retinaface import RetinaFace
from tqdm import tqdm
from IPython.display import Video

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load GazeLLE model
model, transform = torch.hub.load('fkryan/gazelle', 'gazelle_dinov2_vitl14_inout')
model.eval()
model.to(device)

In [ ]:
from google.colab import files
uploaded = files.upload()
video_path = next(iter(uploaded))


In [ ]:
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, fps, (width, height))

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

def visualize_all(pil_image, heatmaps, bboxes, inout_scores, inout_thresh=0.5):

    colors = ['lime', 'tomato', 'cyan', 'fuchsia', 'yellow']
    overlay_image = pil_image.convert("RGBA")
    draw = ImageDraw.Draw(overlay_image)

    width, height = pil_image.size

    for i in range(len(bboxes)):
        xmin, ymin, xmax, ymax = bboxes[i]
        color = colors[i % len(colors)]

        draw.rectangle(
            [xmin * width, ymin * height, xmax * width, ymax * height],
            outline=color,
            width=3
        )

        if inout_scores is not None:
            inout_score = inout_scores[i]

            text = f"in-frame: {inout_score:.2f}"

            draw.text(
                (xmin * width, ymax * height + 5),
                text,
                fill=color,
                font=ImageFont.load_default()
            )

        if inout_scores is not None and inout_score > inout_thresh:

            heatmap = heatmaps[i]
            heatmap_np = heatmap.detach().cpu().numpy()

            max_index = np.unravel_index(
                np.argmax(heatmap_np),
                heatmap_np.shape
            )

            gaze_target_x = max_index[1] / heatmap_np.shape[1] * width
            gaze_target_y = max_index[0] / heatmap_np.shape[0] * height

            bbox_center_x = ((xmin + xmax) / 2) * width
            bbox_center_y = ((ymin + ymax) / 2) * height

            draw.ellipse(
                [(gaze_target_x-5, gaze_target_y-5),
                 (gaze_target_x+5, gaze_target_y+5)],
                fill=color
            )

            draw.line(
                [(bbox_center_x, bbox_center_y),
                 (gaze_target_x, gaze_target_y)],
                fill=color,
                width=3
            )

    # display the visualization
    display(overlay_image)

    return overlay_image

In [ ]:
# --- MAIN LOOP ---
for _ in tqdm(range(frame_count)):
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(rgb_frame)

    try:
        resp = RetinaFace.detect_faces(np.array(pil_image))
        if not resp:
            out.write(frame)
            continue

        bboxes = [resp[key]['facial_area'] for key in resp.keys()]
        norm_bboxes = [[np.array(bbox) / np.array([width, height, width, height]) for bbox in bboxes]]

        img_tensor = transform(pil_image).unsqueeze(0).to(device)
        input_data = {"images": img_tensor, "bboxes": norm_bboxes}

        with torch.no_grad():
            output = model(input_data)

        overlaid_img = visualize_all(pil_image, output['heatmap'][0], norm_bboxes[0], output['inout'][0])
        out_frame = cv2.cvtColor(np.array(overlaid_img), cv2.COLOR_RGBA2BGR)
        out.write(out_frame)

    except Exception as e:
        print(f"Error in frame: {e}")
        out.write(frame)

cap.release()
out.release()

# --- DISPLAY RESULT ---
Video("output.mp4", embed=True)

In [ ]:
import os

# find the first mp3 file in the current directory
audio_file = None

for f in os.listdir():
    if f.endswith(".mp3"):
        audio_file = f
        break

if audio_file is None:
    print("No MP3 file found")
else:
    print("Audio file:", audio_file)
    print("File exists:", os.path.exists(audio_file))
    print("Size:", os.path.getsize(audio_file), "bytes")

In [ ]:
from google.colab import files
files.download("output.mp4")

In [ ]:
import os
print(os.listdir())

In [ ]:
!ls

In [ ]:
import os
os.path.exists("output_final.mp4")

In [ ]:
import os
print(os.listdir())

In [ ]:
import os
print("Video exists:", os.path.exists("output.mp4"))
print("Size:", os.path.getsize("output.mp4") if os.path.exists("output.mp4") else "missing")

In [ ]:
out = cv2.VideoWriter('output.mp4', fourcc, fps, (width, height))

print("Writer opened:", out.isOpened())

In [ ]:
result_img = visualize_all(
    pil_image,
    output['heatmap'][0],
    norm_bboxes,
    output['inout'][0]
)